# Stage 4a: Self-Play Data Generation

**Goal:** Generate 4 hours of self-play conversations from the full Moshi teacher,
capturing teacher targets (sparse logits + hard labels) for offline distillation.

**Run on:** Colab Pro (A100 preferred, L4 acceptable) or Kaggle

**Time estimate:**
- A100: ~2 hours
- L4: ~5.5 hours  
- T4: ~11 hours (needs 1 session)

**Output:** ~480 conversations saved as `.npz` files to GDrive

## Cell 1: Session Startup

In [1]:
# === SESSION STARTUP ===
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess, sys, os

# Fetch GitHub token from Colab Secrets
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("Warning: GITHUB_TOKEN not found in Colab Secrets.")
    GITHUB_TOKEN = ""

REPO_OWNER = "RidwanHR5"
REPO_NAME = "moshilite"

# Construct URL with auth token
if GITHUB_TOKEN:
    REPO = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
else:
    REPO = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
REPO_DIR = "/content/moshilite"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", REPO_DIR, "-q"], check=True)

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
PyTorch: 2.9.1+cu128


## Cell 2: Load Moshi Teacher

In [2]:
import torch
import json
from pathlib import Path
from moshi.models import loaders
from moshi.models.lm import LMGen

WEIGHTS_DIR = "/content/drive/MyDrive/moshilite/upstream_weights/moshiko"

# Detect GPU capability for dtype
gpu_name = torch.cuda.get_device_name(0).lower()
if any(x in gpu_name for x in ["a100", "h100", "l4", "l40"]):
    DTYPE = torch.bfloat16
    print(f"Using bfloat16 (detected {gpu_name})")
else:
    DTYPE = torch.float16
    print(f"Using float16 (detected {gpu_name})")

print("Loading Moshi teacher...")
weights_path = Path(WEIGHTS_DIR)

if not weights_path.exists():
    raise FileNotFoundError(f"Cannot find weights dir: {weights_path}")

print(f"Loading local weights from directory: {weights_path}")
config_path = weights_path / "config.json"
moshi_name = "model.safetensors"

# 1. Try to load config if it exists
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    print("   found config.json")
    moshi_name = config.get("moshi_name", moshi_name)
else:
    print("   no config.json found — using default Moshiko architecture")
    config = None  # None tells get_moshi_lm() to use the 16-codebook default

model_path = weights_path / moshi_name

# 2. If the default 'model.safetensors' doesn't exist, try to find any safetensors file
if not model_path.exists():
    sf_files = list(weights_path.glob("*.safetensors"))
    if not sf_files:
        raise FileNotFoundError(f"Could not find any .safetensors file in {weights_path}")
    model_path = sf_files[0]
    print(f"   auto-detected weights file: {model_path.name}")

# Direct load
lm_model = loaders.get_moshi_lm(
    model_path, lm_kwargs=config, device="cuda", dtype=DTYPE
)

print(f"✅ Model loaded:")
print(f"   n_q={lm_model.n_q}, dep_q={lm_model.dep_q}, dim={lm_model.dim}")
print(f"   num_codebooks={lm_model.num_codebooks}")
print(f"   card={lm_model.card}, text_card={lm_model.text_card}")
print(f"   params: {sum(p.numel() for p in lm_model.parameters()) / 1e9:.2f}B")
print(f"   VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


Using bfloat16 (detected nvidia a100-sxm4-40gb)
Loading Moshi teacher...
Loading local weights from directory: /content/drive/MyDrive/moshilite/upstream_weights/moshiko
   no config.json found — using default Moshiko architecture
✅ Model loaded:
   n_q=16, dep_q=8, dim=4096
   num_codebooks=17
   card=2048, text_card=32000
   params: 7.69B
   VRAM: 15.4 GB


## Cell 3: Quick Sanity Test (1 conversation)

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — CHANGE THESE PER PERSON / SESSION
# ============================================================
PERSON_ID = "A"              # Unique per person: A, B, C, D...
SESSION_NUM = 0              # Increment each session: 0, 1, 2...
CONVOS_PER_SESSION = 40      # ~40 on T4 (10hr), ~120 on A100 (7hr)

# Auto-computed — DO NOT EDIT
BATCH_ID = f"batch_{PERSON_ID}_{SESSION_NUM:02d}"   # e.g. batch_A_00
STEPS_PER_CONV = 3750                                # 5 minutes @ 12.5 Hz
OUTPUT_DIR = "/content/drive/MyDrive/moshilite/self_play_data/conversations"

print(f"👤 Person: {PERSON_ID}")
print(f"📦 Batch:  {BATCH_ID}")
print(f"🎯 Target: {CONVOS_PER_SESSION} conversations × 5 min = {CONVOS_PER_SESSION * 5 / 60:.1f} hours")
print(f"📂 Output: {OUTPUT_DIR}/{BATCH_ID}/")

# Quick sanity test (short conversation to measure speed)
import time
from moshi.models.lm import LMGen
from moshilite.self_play.generator import generate_conversation

lm_gen = LMGen(
    lm_model,
    use_sampling=True,
    temp=0.8,
    temp_text=0.7,
    top_k=250,
    top_k_text=25,
)

print("\nGenerating test conversation (50 steps ≈ 4 seconds)...")
t0 = time.time()
record = generate_conversation(
    lm_gen=lm_gen,
    num_steps=50,
    seed_type="noise",
    seed_index=0,
    top_k_logits=50,
    conv_id="test_000",
    card=lm_model.card,
    device="cuda",
)
elapsed = time.time() - t0
ms_per_step = (elapsed / 50) * 1000

print(f"\n✅ Test passed: {record.num_valid_steps} valid steps in {elapsed:.1f}s ({ms_per_step:.0f} ms/step)")

# Estimate session time
est_hours = (STEPS_PER_CONV * CONVOS_PER_SESSION * 1.5 * ms_per_step / 1000) / 3600  # 1.5x for rejections
print(f"⏱️  Estimated session time: {est_hours:.1f} hours (with ~33% rejection overhead)")


Generating test conversation (50 steps ≈ 4 seconds)...

✅ Test conversation generated:
   Valid steps: 49
   Text tokens shape: (49,)
   Audio tokens shape: (8, 49)
   Text logits shape: (49, 50)
   Audio CB0 logits shape: (49, 50)
   Time: 6.0s (121 ms/step)

📊 Estimated time for 480 conversations: 6.0 hours
   (at 121 ms/step)


## Cell 4: Generate Full Batch

In [ ]:
from moshilite.self_play.generator import generate_batch

print(f"🚀 Starting generation: {CONVOS_PER_SESSION} conversations, "
      f"{STEPS_PER_CONV} steps each (5 min)")
print(f"📦 Batch: {BATCH_ID}")
print(f"📂 Output: {OUTPUT_DIR}/{BATCH_ID}/")

stats = generate_batch(
    lm_gen=lm_gen,
    num_conversations=CONVOS_PER_SESSION,
    steps_per_conversation=STEPS_PER_CONV,
    top_k_logits=50,
    output_dir=OUTPUT_DIR,
    batch_id=BATCH_ID,
    start_index=0,
    card=lm_model.card,
    device="cuda",
    repetition_penalty=1.3,
    rep_window=50,
)


Starting single 375-step diagnostic run...

Time elapsed: 10.0 seconds
Accepted: False
Rejection Reason: text_repetition (0.85 > 0.5)
Details: {'text_4gram_repetition': 0.8544474393530997}


In [4]:
import time
import numpy as np
from collections import Counter
from moshilite.self_play.generator import generate_conversation
from moshilite.self_play.quality_filter import filter_conversation

seed_types = ["noise", "acoustic", "silence"]

for i, seed in enumerate(seed_types):
    print(f"{'='*60}")
    print(f"Conversation {i+1}/3 (Seed: '{seed}')")

    record = generate_conversation(
        lm_gen=lm_gen,
        num_steps=375,
        seed_type=seed,
        seed_index=100 + i,
        top_k_logits=50,
        conv_id=f"diag_{i}",
        card=lm_model.card,
        device="cuda",
    )

    quality = filter_conversation(
        text_tokens=record.text_tokens,
        audio_tokens=record.audio_tokens,
        num_valid_steps=record.num_valid_steps,
    )

    txt = record.text_tokens[:record.num_valid_steps]
    aud = record.audio_tokens[:, :record.num_valid_steps]

    # Text token analysis
    txt_counts = Counter(txt.tolist())
    pad_ratio = txt_counts.get(0, 0) / len(txt)
    unique_text = len(txt_counts)
    top5_text = txt_counts.most_common(5)

    # Audio CB0 analysis
    cb0 = aud[0]
    aud_counts = Counter(cb0.tolist())
    unique_audio = len(aud_counts)
    top5_audio = aud_counts.most_common(5)

    status = '✅ ACCEPTED' if quality.accepted else '❌ REJECTED'
    print(f"  Status:        {status}")
    if not quality.accepted:
        print(f"  Reason:        {quality.rejection_reason}")
    print(f"  Valid steps:   {record.num_valid_steps}")
    print(f"  --- Text Tokens ---")
    print(f"  Unique tokens: {unique_text}")
    print(f"  Pad ratio:     {pad_ratio:.1%}")
    print(f"  Top-5 tokens:  {top5_text}")
    print(f"  --- Audio CB0 ---")
    print(f"  Unique tokens: {unique_audio}")
    print(f"  Top-5 tokens:  {top5_audio}")
    print(f"  Rep ratio:     {quality.repetition_ratio:.3f}")
    print(f"  Silence ratio: {quality.silence_ratio:.3f}")
    print(f"  Details:       {quality.details}")
    print()


Conversation 1/3 (Seed: 'noise')
  Status:        ✅ ACCEPTED
  Valid steps:   374
  --- Text Tokens ---
  Unique tokens: 25
  Pad ratio:     4.5%
  Top-5 tokens:  [(3, 328), (0, 17), (261, 2), (330, 2), (293, 2)]
  --- Audio CB0 ---
  Unique tokens: 84
  Top-5 tokens:  [(2029, 91), (948, 69), (1316, 64), (175, 27), (222, 8)]
  Rep ratio:     0.836
  Silence ratio: 0.000
  Details:       {'text_4gram_repetition': 0.8355795148247979, 'audio_cb0_4gram_repetition': 0.6361185983827493, 'silence_ratio': 0.0}

Conversation 2/3 (Seed: 'acoustic')
  Status:        ❌ REJECTED
  Reason:        text_repetition (0.95 > 0.92)
  Valid steps:   374
  --- Text Tokens ---
  Unique tokens: 9
  Pad ratio:     1.1%
  Top-5 tokens:  [(3, 363), (0, 4), (11725, 1), (261, 1), (452, 1)]
  --- Audio CB0 ---
  Unique tokens: 32
  Top-5 tokens:  [(1316, 222), (2029, 82), (948, 21), (1926, 7), (752, 6)]
  Rep ratio:     0.951
  Silence ratio: 0.000
  Details:       {'text_4gram_repetition': 0.9514824797843666}

Con

In [5]:
from moshilite.self_play.generator import generate_batch

OUTPUT_DIR = "/content/drive/MyDrive/moshilite/self_play_data/conversations"
BATCH_ID = "batch_000"         # Change for each session
NUM_CONVERSATIONS = 480        # Target: 4 hours of audio
STEPS_PER_CONV = 375           # 30 seconds @ 12.5 Hz
START_INDEX = 0                # Change for multi-session (e.g., 480 for 2nd session)

print(f"Starting generation: {NUM_CONVERSATIONS} conversations, "
      f"{STEPS_PER_CONV} steps each")
print(f"Output: {OUTPUT_DIR}/{BATCH_ID}/")

stats = generate_batch(
    lm_gen=lm_gen,
    num_conversations=NUM_CONVERSATIONS,
    steps_per_conversation=STEPS_PER_CONV,
    top_k_logits=50,
    output_dir=OUTPUT_DIR,
    batch_id=BATCH_ID,
    start_index=START_INDEX,
    card=lm_model.card,
    device="cuda",
)

Starting generation: 480 conversations, 375 steps each
Output: /content/drive/MyDrive/moshilite/self_play_data/conversations/batch_000/


Generating batch_000: 100%|██████████| 480/480 [1:55:52<00:00, 14.48s/it]


✅ Batch batch_000: 480 conversations, 3.99 hours, 115.9 min wall time, 68.3% acceptance rate


## Cell 5: Verify & Report

In [6]:
import json
from pathlib import Path
import numpy as np

batch_dir = Path(OUTPUT_DIR) / BATCH_ID

# Load batch metadata
with open(batch_dir / "batch_meta.json") as f:
    meta = json.load(f)

print(f"📊 Batch {BATCH_ID} Summary:")
print(f"   Accepted: {meta['accepted']}")
print(f"   Rejected: {meta['rejected']}")
print(f"   Acceptance rate: {meta['acceptance_rate']:.1%}")
print(f"   Total audio: {meta['total_audio_hours']:.2f} hours")
print(f"   Wall time: {meta['total_wall_time_s'] / 60:.1f} min")
print(f"   Rejection reasons: {meta.get('rejection_reasons', {})}")

# Verify a few files
npz_files = sorted(batch_dir.glob("*.npz"))
print(f"\n📂 {len(npz_files)} .npz files on disk")

if npz_files:
    sample = np.load(str(npz_files[0]))
    print(f"\nSample file contents ({npz_files[0].name}):")
    for k in sample.files:
        arr = sample[k]
        print(f"   {k}: shape={arr.shape}, dtype={arr.dtype}")

    total_size_mb = sum(f.stat().st_size for f in npz_files) / 1e6
    print(f"\nTotal disk usage: {total_size_mb:.1f} MB")

📊 Batch batch_000 Summary:
   Accepted: 480
   Rejected: 223
   Acceptance rate: 68.3%
   Total audio: 3.99 hours
   Wall time: 115.9 min
   Rejection reasons: {'text_repetition (0.95 > 0.92)': 142, 'text_repetition (0.96 > 0.92)': 42, 'text_repetition (0.94 > 0.92)': 32, 'text_repetition (0.93 > 0.92)': 6, 'text_repetition (0.92 > 0.92)': 1}

📂 480 .npz files on disk

Sample file contents (conv_000000.npz):
   text_tokens: shape=(374,), dtype=int16
   audio_tokens: shape=(8, 374), dtype=int16
   text_logits_vals: shape=(374, 50), dtype=float16
   text_logits_idxs: shape=(374, 50), dtype=int32
   audio_cb0_logits_vals: shape=(374, 50), dtype=float16
   audio_cb0_logits_idxs: shape=(374, 50), dtype=int32
   user_audio_tokens: shape=(8, 374), dtype=int16
   num_valid_steps: shape=(1,), dtype=int32

Total disk usage: 3.5 MB


New Cell 6 (Code) — add at the end for main person to merge & verify:


In [ ]:
# ============================================================
# 🔗 MERGE & VERIFY (Run by lead person after collecting batches)
# ============================================================
from pathlib import Path
import json

data_dir = Path("/content/drive/MyDrive/moshilite/self_play_data/conversations")
all_npz = sorted(data_dir.rglob("*.npz"))
all_npz = [f for f in all_npz if "rejected" not in f.parts]

# Group by batch
batches = {}
for f in all_npz:
    batch = f.parent.name
    batches.setdefault(batch, []).append(f)

print(f"📊 DATASET SUMMARY")
print(f"{'='*60}")
total_hours = 0
for batch, files in sorted(batches.items()):
    hours = len(files) * 5 / 60  # 5 min per conversation
    total_hours += hours
    meta_path = data_dir / batch / "batch_meta.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        acc_rate = f"{meta.get('acceptance_rate', 0):.0%}"
    else:
        acc_rate = "n/a"
    print(f"  {batch:20s}: {len(files):4d} convos × 5 min = {hours:5.1f} hrs  (acceptance: {acc_rate})")

print(f"{'='*60}")
print(f"  TOTAL: {len(all_npz)} conversations = {total_hours:.1f} hours")
print(f"\n✅ Ready for training!" if total_hours >= 50 else
      f"\n⏳ Need {50 - total_hours:.0f} more hours to reach 50-hour minimum")
